# Dollar Neutral: V Long / MA Short (1:1 Hedge Ratio)

This notebook demonstrates the `DollarNeutral` strategy using Visa (V) as the long leg
and Mastercard (MA) as the short leg, with a **1:1 hedge ratio** (long $1 V, short $1 MA).

BIL (SPDR Bloomberg 1-3 Month T-Bill ETF) absorbs collateral and residual cash.

**Data**: Yahoo Finance (auto-adjusted)  
**Rebalance**: Monthly mid-month (`month_mid` = 15th of each month)  
**Tolerance**: 5% imbalance before forced rebalance

**Rationale**: V and MA are both in the Payments industry with extremely high correlation.
Longing one and shorting the other should cancel out market volatility and achieve
true market neutral state (Beta ≈ 0).

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio.helpers.data import YFinance
from tiportfolio import (
    DollarNeutral, FixRatio, Schedule, ScheduleBasedEngine,
    compare_strategies, plot_strategy_comparison_interactive,
    rebalance_decisions_table,
)

enable_data_source_cache("tiportfolio", cache_dir=".cache")

LONG   = "V"
SHORT  = "MA"
CASH   = "BIL"
RATIO  = 1.0             # long $1 V, short $1 MA (50/50 for initial test)
START  = "2023-01-01"   # Extended start for better analysis
END    = "2024-12-31"
INITIAL_VALUE = 10_000

# Symmetric book sizes (50/50)
LONG_BS  = 1.0 / (1.0 + RATIO)   # = 0.5
SHORT_BS = RATIO / (1.0 + RATIO)  # = 0.5
print(f"long_book_size={LONG_BS:.4f}  short_book_size={SHORT_BS:.4f}  ratio={SHORT_BS/LONG_BS:.4f}")

long_book_size=0.5000  short_book_size=0.5000  ratio=1.0000


In [2]:
yf = YFinance(auto_adjust=True)
df = yf.query([LONG, SHORT, CASH], start_date=START, end_date=END)

prices = {}
for symbol in df["symbol"].unique():
    sub = df[df["symbol"] == symbol].set_index("date")[["open", "high", "low", "close"]]
    prices[symbol] = sub

print(f"Loaded {len(prices)} symbols")
for sym, d in prices.items():
    print(f"  {sym}: {len(d)} rows  {d.index[0].date()} → {d.index[-1].date()}")

Loading bar data...


[*********************100%***********************]  3 of 3 completed


Loaded bar data: 0:00:03 

Loaded 3 symbols
  BIL: 501 rows  2023-01-03 → 2024-12-30
  MA: 501 rows  2023-01-03 → 2024-12-30
  V: 501 rows  2023-01-03 → 2024-12-30


In [3]:
strategy = DollarNeutral(
    long_weights={LONG: 1.0},
    short_weights={SHORT: 1.0},
    cash_symbol=CASH,
    long_book_size=LONG_BS,
    short_book_size=SHORT_BS,
    tolerance=0.05,
)

engine = ScheduleBasedEngine(
    allocation=strategy,
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)

result = engine.run(
    symbols=[LONG, SHORT, CASH],
    start=START, end=END,
    prices_df=prices,
)
print(result.summary())

Backtest Summary
----------------
Sharpe Ratio:        0.1230
Sortino Ratio:       0.1700
MAR Ratio:           0.8118
CAGR:                4.66%
Max Drawdown:        5.73%
Kelly Leverage:      2.0112
Mean Excess Return:  0.0075
Final Value:         10,947.99
Total Fee:           0.06
Rebalances:          24


In [4]:
fig = result.plot_portfolio(mark_rebalances=True)
fig.show()

In [5]:
decisions_df = rebalance_decisions_table(result)
decisions_df

,date,equity_before,equity_after,fee_paid,V_price,V_qty_before,V_trade_qty,V_qty_after,V_value_after,MA_price,MA_qty_before,MA_trade_qty,MA_qty_after,MA_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after
0,2023-01-13 00:00:00+00:00,9965.863,9965.863,0.00,217.626,24.711,0.000,24.711,5377.791,369.021,-14.695,0.000,-14.695,-5422.858,79.364,126.139,-0.000,126.139,10010.931
1,2023-02-15 00:00:00+00:00,10237.474,10237.474,0.00,223.781,24.711,0.000,24.711,5529.880,363.242,-14.695,0.000,-14.695,-5337.945,79.639,126.139,0.000,126.139,10045.539
2,2023-03-15 00:00:00+00:00,10284.671,10284.671,0.00,211.513,24.711,0.000,24.711,5226.718,342.056,-14.695,0.000,-14.695,-5026.598,79.948,126.139,0.000,126.139,10084.552
3,2023-04-14 00:00:00+00:00,10388.663,10388.663,0.00,228.766,24.711,0.000,24.711,5653.078,366.558,-14.695,0.000,-14.695,-5386.673,80.247,126.139,0.000,126.139,10122.259
4,2023-05-15 00:00:00+00:00,10246.235,10246.235,0.00,228.027,24.711,0.000,24.711,5634.813,377.365,-14.695,0.000,-14.695,-5545.483,80.521,126.139,0.000,126.139,10156.905
5,2023-06-15 00:00:00+00:00,10195.524,10195.524,0.00,221.524,24.711,-0.000,24.711,5474.102,373.064,-14.695,0.000,-14.695,-5482.278,80.892,126.139,-0.000,126.139,10203.700
6,2023-07-14 00:00:00+00:00,10298.248,10298.248,0.00,238.165,24.711,0.000,24.711,5885.319,396.737,-14.695,0.000,-14.695,-5830.157,81.205,126.139,0.000,126.139,10243.086
7,2023-08-15 00:00:00+00:00,10388.341,10388.341,0.00,235.345,24.711,0.000,24.711,5815.637,388.793,-14.695,0.000,-14.695,-5713.412,81.546,126.139,0.000,126.139,10286.115
8,2023-09-15 00:00:00+00:00,10179.682,10179.682,0.00,236.562,24.711,-0.000,24.711,5845.706,408.368,-14.695,0.000,-14.695,-6001.074,81.934,126.139,0.000,126.139,10335.051
9,2023-10-16 00:00:00+00:00,10380.884,10380.884,0.00,235.580,24.711,0.000,24.711,5821.456,395.980,-14.695,0.000,-14.695,-5819.027,82.278,126.139,0.000,126.139,10378.454


## Rolling Book Composition

In [6]:
fig_book = result.plot_rolling_book_composition(universe=[LONG, SHORT])
fig_book.show()

book_df = result.book_composition_table(universe=[LONG, SHORT])
print(book_df.to_string())

               V     MA
date                   
2023-01-13  Long  Short
2023-02-15  Long  Short
2023-03-15  Long  Short
2023-04-14  Long  Short
2023-05-15  Long  Short
2023-06-15  Long  Short
2023-07-14  Long  Short
2023-08-15  Long  Short
2023-09-15  Long  Short
2023-10-16  Long  Short
2023-11-15  Long  Short
2023-12-15  Long  Short
2024-01-16  Long  Short
2024-02-15  Long  Short
2024-03-15  Long  Short
2024-04-15  Long  Short
2024-05-15  Long  Short
2024-06-14  Long  Short
2024-07-15  Long  Short
2024-08-15  Long  Short
2024-09-16  Long  Short
2024-10-15  Long  Short
2024-11-15  Long  Short
2024-12-16  Long  Short


## Comparison to Baselines

We compare the dollar-neutral spread against three single-leg alternatives:
- **Long V**: 100% V, buy-and-hold the payments leader
- **Short MA** (synthetic): 100% BIL + MA at −100% — pure short payments exposure
- **50/50 V+BIL**: half in V, half in T-bills — a simple risk-managed long

In [7]:
# Baseline 1: 100% long V
engine_v = ScheduleBasedEngine(
    allocation=FixRatio(weights={LONG: 1.0}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_long_v = engine_v.run(
    symbols=[LONG], start=START, end=END, prices_df={LONG: prices[LONG]}
)
print("Long V only")
print(result_long_v.summary())

Long V only
Backtest Summary
----------------
Sharpe Ratio:        1.1831
Sortino Ratio:       1.6737
MAR Ratio:           1.9589
CAGR:                24.39%
Max Drawdown:        12.45%
Kelly Leverage:      7.2806
Mean Excess Return:  0.1923
Final Value:         15,441.59
Total Fee:           0.00
Rebalances:          0


In [8]:
# Baseline 2: 50% V / 50% BIL
engine_half = ScheduleBasedEngine(
    allocation=FixRatio(weights={LONG: 0.5, CASH: 0.5}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_half = engine_half.run(
    symbols=[LONG, CASH], start=START, end=END,
    prices_df={LONG: prices[LONG], CASH: prices[CASH]},
)
print("50% V / 50% BIL")
print(result_half.summary())

50% V / 50% BIL
Backtest Summary
----------------
Sharpe Ratio:        1.2418
Sortino Ratio:       1.7598
MAR Ratio:           2.6644
CAGR:                14.69%
Max Drawdown:        5.51%
Kelly Leverage:      15.2886
Mean Excess Return:  0.1009
Final Value:         13,137.41
Total Fee:           0.14
Rebalances:          24


In [9]:
compare_strategies(
    result, result_long_v, result_half,
    names=["DollarNeutral V/MA", "Long V", "50/50 V+BIL"],
)

,DollarNeutral V/MA,Long V,50/50 V+BIL,better
metric,,,,
sharpe_ratio,0.122960,1.183142,1.241790,50/50 V+BIL
sortino_ratio,0.169952,1.673696,1.759832,50/50 V+BIL
mar_ratio,0.811817,1.958870,2.664442,50/50 V+BIL
cagr,0.046555,0.243942,0.146939,Long V
max_drawdown,0.057346,0.124532,0.055148,50/50 V+BIL


In [10]:
plot_strategy_comparison_interactive(
    result, result_long_v, result_half,
)

## Portfolio Beta

Track portfolio beta over time vs SPY.

In [11]:
# Plot portfolio beta over time
fig_beta = result.plot_portfolio_beta()
fig_beta.show()

Loading bar data...


[*********************100%***********************]  1 of 1 completed

Loaded bar data: 0:00:01 



## Interpretation
### 1. Empirical Results & Reality Check
*   **Risk Mitigation (Success):** The strategy successfully achieved its primary goal of risk reduction. The Max Drawdown was contained at ~5.7%, which is exceptionally stable compared to the standalone Long V strategy.
*   **Performance Drag (Failure):** The strategy severely underperformed in risk-adjusted metrics (Sharpe Ratio: 0.12, CAGR: 4.6%). It was outperformed by the simple `50/50 V+BIL` baseline (Sharpe: 1.24).
*   **The "Why":** The correlation between V and MA during this period was *too strong* without sufficient mean-reverting divergence. While V did outperform MA slightly, the spread was too narrow. The profit from the long leg barely covered the drag from the short leg, resulting in a net return that barely beat the risk-free rate (BIL).

### 2. Summary
This V/MA experiment proves that the `ScheduleBasedEngine` and the Dollar Neutral mechanics work flawlessly. However, it also proves that **relying on fundamental similarity (industry duopolies) is not enough to generate Alpha in a Dollar Neutral setup.**

### 3. Optimization
1.  **Statistical Arbitrage:** Implementing a scanner to find pairs based on statistical metrics (e.g., Cointegration, Z-score of the spread) rather than just industry labels.
2.  **Dynamic Sizing:** Adjusting the hedge ratio dynamically based on rolling volatility (Beta-neutral) rather than a fixed 1:1 dollar-neutral ratio.